![Cabecera](../../../Sprint_11/assets/cabecera_thebridge.png)

# Workout — Memoria y checkpoints

## Objetivo

Que el grafo **recuerde** entre llamadas a `invoke` usando:

- `MemorySaver` — guarda un *checkpoint* (foto del estado) en RAM tras cada paso.
- `thread_id` — identifica **qué conversación** es. 

```text
Sin checkpointer:  invoke → olvida → invoke → olvida
Con checkpointer:  invoke (thread=ana) → guarda → invoke (thread=ana) → recuerda
```

Checkpointer en LangGraph es un mecanismo de persistencia que guarda una instantánea (snapshot) del estado del grafo en cada super-step (paso de ejecución), permitiendo memoria a corto plazo, control humano (human-in-the-loop), viajes en el tiempo y tolerancia a fallos. 

Veremos: dos turnos en el mismo hilo, un hilo distinto que no ve a Ana, e inspección con `get_state`.

## Índice

1. Setup
2. Chatbot mínimo con historial
3. Dos turnos en el mismo `thread_id`
4. Otro `thread_id` no ve al primer usuario
5. Inspeccionar el State
6. Para llevarnos

**Requisito:** `GEMINI_API_KEY`


## 1. Setup

Descomenta la celda de `%pip` solo la primera vez o si falta algún paquete.

In [ ]:
# %pip install -qU langgraph langchain-google-genai langchain-core python-dotenv

In [1]:
# 1) Intentamos leer GEMINI_API_KEY desde un fichero .env
# 2) Si no está, la pedimos oculta con getpass (no se imprime en pantalla)
import os
from getpass import getpass

from dotenv import load_dotenv

load_dotenv()

if not os.getenv("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("Pega tu GEMINI_API_KEY: ")

API_KEY = os.environ["GEMINI_API_KEY"]
print("GEMINI_API_KEY:", "sí" if API_KEY else "no")

GEMINI_API_KEY: sí


## 2. Chatbot mínimo con historial de mensajes

Montamos un grafo **muy corto** (un solo nodo) para centrarnos en la memoria:

```text
START → chat → END
```

| Pieza | Para qué |
|-------|----------|
| `messages` + `add_messages` | El historial **acumula** (igual que en ReAct) |
| `MemorySaver` al `compile` | Guarda el estado entre `invoke` |
| `thread_id` (en la siguiente sección) | Dice *qué* hilo cargar/guardar |

Ojo: `add_messages` acumula mensajes **dentro** del State. Sin `MemorySaver`, eso sirve en un solo `invoke`, pero **entre** un `invoke` y el siguiente el grafo olvidaría. El checkpointer es lo que hace recordar de una llamada a otra.

El nodo `chat` solo llama al LLM y **devuelve el mensaje nuevo**; LangGraph lo añade a la lista.

In [2]:
from typing import Annotated, TypedDict

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

# Modelo (mismo estilo que el notebook ReAct)
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=API_KEY,
    temperature=0,
)


# Estado: solo el historial de chat.
# add_messages = si el nodo devuelve mensajes nuevos, se AÑADEN (no se pisa la lista).
class EstadoChat(TypedDict):
    messages: Annotated[list, add_messages]  # add_messages → acumula


def nodo_chat(estado: EstadoChat) -> dict:
    """Un turno: system + historial → LLM → solo el mensaje nuevo."""
    msgs = [SystemMessage(content="Eres un asistente breve y claro.")] + list(
        estado["messages"]
    )
    respuesta = llm.invoke(msgs)
    return {"messages": [respuesta]}  # LangGraph lo concatena al historial


# Grafo lineal: una sola visita al LLM por invoke
grafo = StateGraph(EstadoChat)
grafo.add_node("chat", nodo_chat)
grafo.add_edge(START, "chat")
grafo.add_edge("chat", END)

# MemorySaver = checkpoints en RAM (se pierden al reiniciar el kernel)
app = grafo.compile(checkpointer=MemorySaver())
print("Compilado con MemorySaver")

Compilado con MemorySaver


## 3. Dos turnos en el mismo `thread_id`

Con checkpointer, **cada** `invoke` debe llevar un `config` con `thread_id`.

Receta fija (cópiala; no inventes la forma):

```python
config = {"configurable": {"thread_id": "ana-001"}}
```

Aquí usamos el mismo id (`ana-001`) dos veces:

1. Ana se presenta (nombre + gusto musical).
2. Preguntamos cómo se llama — el modelo debería **recordarlo** porque el historial vive en ese hilo.

Si olvidaras el `config`, LangGraph no sabría qué checkpoint cargar.

In [3]:
# Mismo thread_id = misma conversación (mismo checkpoint)
config_ana = {"configurable": {"thread_id": "ana-001"}}

# Turno 1: Ana da datos nuevos
r1 = app.invoke(
    {"messages": [HumanMessage(content="Hola, me llamo Ana y me gusta el jazz.")]},
    config=config_ana,
)
print("Turno 1:", r1["messages"][-1].content)

# Turno 2: misma Ana — el State ya tiene el turno 1
r2 = app.invoke(
    {"messages": [HumanMessage(content="¿Cómo me llamo y qué música me gusta?")]},
    config=config_ana,
)
print("Turno 2:", r2["messages"][-1].content)
# Esperado: varios mensajes (Human + AI del turno 1, Human + AI del turno 2, …)
print("Mensajes en el hilo:", len(r2["messages"]))

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Turno 1: Hola Ana, ¡qué buen gusto con el jazz!
Turno 2: Te llamas Ana y te gusta el jazz.
Mensajes en el hilo: 4


## 4. Otro `thread_id` no ve a Ana

Cambiamos a `bob-001`. Es **otra** conversación: el checkpointer de Ana no se carga.

Bob pregunta "¿Cómo me llamo?" sin haberse presentado → el modelo **no** debería responder "Ana".

Idea clave: `thread_id` = aislamiento entre usuarios / sesiones (en Streamlit suele ser el id de la sesión de chat).

In [4]:
# thread_id distinto = conversación vacía (no ve el historial de ana-001)
config_bob = {"configurable": {"thread_id": "bob-001"}}

r_bob = app.invoke(
    {"messages": [HumanMessage(content="¿Cómo me llamo?")]},
    config=config_bob,
)
print("Hilo bob:", r_bob["messages"][-1].content)

Hilo bob: No lo sé. Soy un programa de ordenador y no tengo acceso a tu información personal.


## 5. Inspeccionar el State

`get_state(config)` lee el **último checkpoint** de ese `thread_id` sin llamar al LLM.

Primero el hilo de Ana (varios mensajes). Justo debajo, el de Bob (solo su turno).

In [5]:
# Checkpoint del hilo de Ana (sin nuevo invoke al LLM)
snap_ana = app.get_state(config_ana)
msgs_ana = snap_ana.values.get("messages", [])

print(f"=== Hilo ana-001 ({len(msgs_ana)} mensajes) ===")
for i, m in enumerate(msgs_ana):
    print(i, type(m).__name__, m.content)

=== Hilo ana-001 (4 mensajes) ===
0 HumanMessage Hola, me llamo Ana y me gusta el jazz.
1 AIMessage Hola Ana, ¡qué buen gusto con el jazz!
2 HumanMessage ¿Cómo me llamo y qué música me gusta?
3 AIMessage Te llamas Ana y te gusta el jazz.


Mismo patrón con Bob: historial corto y **sin** los mensajes de Ana.

In [6]:
# Checkpoint del hilo de Bob (thread_id distinto → otro State)
snap_bob = app.get_state(config_bob)
msgs_bob = snap_bob.values.get("messages", [])

print(f"=== Hilo bob-001 ({len(msgs_bob)} mensajes) ===")
for i, m in enumerate(msgs_bob):
    print(i, type(m).__name__, m.content)

=== Hilo bob-001 (2 mensajes) ===
0 HumanMessage ¿Cómo me llamo?
1 AIMessage No lo sé. Soy un programa de ordenador y no tengo acceso a tu información personal.


## 6. Para llevarnos

| Pieza | Rol |
|-------|-----|
| `MemorySaver` | Checkpoints en RAM (se pierden al reiniciar el kernel) |
| `thread_id` | Separa conversaciones / usuarios |
| `config={"configurable": {"thread_id": …}}` | Obligatorio en cada `invoke` con checkpointer |
| `add_messages` | Acumula el historial en el State |
| `get_state` | Mira el State del hilo sin llamar al LLM |

Opcional en producción: `SqliteSaver` para persistir en disco (`langgraph-checkpoint-sqlite`).